# Original Mel generation

- We use the same previously labeled audio segments to convert them into Mel spectrograms. In this way, we will have Mels with their respective (multi) labels (TM, MH, VO, MU, EM, SO, BI, GE, SI)

In [9]:
import os
import pandas as pd
import librosa
import numpy as np
from PIL import Image
import matplotlib.cm as cm

# 1. Basic setup

In [11]:
# INPUT
ORG_AUDIO_DIR = r"C:\Users\Acer\Desktop\TSI\U_VNE_CNN\0_labelling-py-R\data\org_audio_pb"
FIN_LABELS  = r"C:\Users\Acer\Desktop\TSI\U_VNE_CNN\0_labelling-py-R\results\fin_labels_pb.csv"

# OUTPUT
ORG_MELS_DIR = r"C:\Users\Acer\Desktop\TSI\U_VNE_CNN\1_mels_generation-py-R\results\org_mels_pb"
os.makedirs(ORG_MELS_DIR, exist_ok=True)

In [3]:
# MELS PARAMETERS
n_fft = 2048
hop_length = 32
n_mels = 128
fmin = 0
fmax = 11025
IMG_SIZE = 224

In [5]:
# AUDIO SEGMENTS

audio_segm = [f for f in os.listdir(ORG_AUDIO_DIR) if f.lower().endswith(".wav")]
print("Found audio segments:", len(audio_segm))
print("-" * 60)

Found audio segments: 5130
------------------------------------------------------------


# 2. Get Mels

In [12]:
# load final labels
df_fin = pd.read_csv(FIN_LABELS)
labelled_files = set(df_fin["filename"].astype(str))

# original counts
total_org_audios = len(audio_segm)

# filter audios to process
audio_segm_filtered = [a for a in audio_segm if a in labelled_files]

print(f"Total audio segments: {total_org_audios}")
print(f"Audio segments laballed: {len(labelled_files)}")
print(f"Audio segments to generate mels for: {len(audio_segm_filtered)}")
print("-" * 40)

Total audio segments: 5130
Audio segments laballed: 5115
Audio segments to generate mels for: 5115
----------------------------------------


In [13]:
# RUN MELS GENERATION

for idx, audio_name in enumerate(audio_segm_filtered):

    print(f"[{idx+1}/{len(audio_segm_filtered)}] Processing:", audio_name)

    input_audio = os.path.join(ORG_AUDIO_DIR, audio_name)
    base_name = os.path.splitext(audio_name)[0]
    output_mels = os.path.join(ORG_MELS_DIR, base_name + ".png")

    try:
        # load audio
        y, sr = librosa.load(input_audio, sr=None, mono=True)

        # Mel spec
        S = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            fmin=fmin,
            fmax=fmax
        )

        # Convert to dB
        S_db = librosa.power_to_db(S, ref=np.max)

        # Normalization
        S_min, S_max = S_db.min(), S_db.max()
        if S_max - S_min < 1e-6:
            S_norm = np.zeros_like(S_db)
        else:
            S_norm = (S_db - S_min) / (S_max - S_min)

        # low and high frequency position
        S_norm = np.flipud(S_norm)

        # color
        cmap = cm.get_cmap("inferno")
        img_array = (cmap(S_norm)[:, :, :3] * 255).astype(np.uint8)

        # image aspect: square
        img = Image.fromarray(img_array)
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        # saving
        img.save(output_mels)

        print("✅ Saved:", output_mels)
        print("-" * 40)

        del S, S_db, S_norm, img_array, img, y

    except Exception as e:
        print("❌ File error:", audio_name)
        print(e)
        print("-" * 40)

print("✅ Processing completed.")

[1/5115] Processing: 001_04_P1.wav


C:\Users\Acer\AppData\Local\Temp\ipykernel_12360\3757567382.py:40: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("inferno")


✅ Saved: C:\Users\Acer\Desktop\TSI\U_VNE_CNN\1_mels_generation-py-R\results\org_mels_pb\001_04_P1.png
----------------------------------------
[2/5115] Processing: 001_04_P2.wav
✅ Saved: C:\Users\Acer\Desktop\TSI\U_VNE_CNN\1_mels_generation-py-R\results\org_mels_pb\001_04_P2.png
----------------------------------------
[3/5115] Processing: 001_05_P2.wav
✅ Saved: C:\Users\Acer\Desktop\TSI\U_VNE_CNN\1_mels_generation-py-R\results\org_mels_pb\001_05_P2.png
----------------------------------------
[4/5115] Processing: 001_06_P1.wav
✅ Saved: C:\Users\Acer\Desktop\TSI\U_VNE_CNN\1_mels_generation-py-R\results\org_mels_pb\001_06_P1.png
----------------------------------------
[5/5115] Processing: 001_09_P1.wav
✅ Saved: C:\Users\Acer\Desktop\TSI\U_VNE_CNN\1_mels_generation-py-R\results\org_mels_pb\001_09_P1.png
----------------------------------------
[6/5115] Processing: 001_10_P2.wav
✅ Saved: C:\Users\Acer\Desktop\TSI\U_VNE_CNN\1_mels_generation-py-R\results\org_mels_pb\001_10_P2.png
--------

# 3. Check

In [14]:
# ORDER AUDIO
audio_files = sorted([
    f for f in os.listdir(ORG_AUDIO_DIR)
    if f.lower().endswith(".wav")
])

# ORDER MELS
mel_files = set([
    os.path.splitext(f)[0]
    for f in os.listdir(ORG_MELS_DIR)
    if f.lower().endswith(".png")
])

# AUDIO SEGMENTS WITHOUT MELS
missing_audio_files = [
    f for f in audio_files
    if os.path.splitext(f)[0] not in mel_files
]

print("✅ Total audio segments:", len(audio_files))
print("✅ Total mels:", len(mel_files))
print("❗ Audio segments without mels:", len(missing_audio_files))

missing_audio_files[:10]

✅ Total audio segments: 5130
✅ Total mels: 5115
❗ Audio segments without mels: 15


['005_06_P2.wav',
 '006_05_P2.wav',
 '006_10_P2.wav',
 '007_28_P2.wav',
 '007_76_P1.wav',
 '010_82_P2.wav',
 '018_07_P1.wav',
 '063_59_P1.wav',
 '070_31_P1.wav',
 '073_04_P1.wav']